# 🔌 API Exploration Lab
Day 2 – Session 2

In this notebook we will authenticate, send API requests, inspect responses, and implement simple robustness patterns.

## 1) Setup & Credentials

In [ ]:
import os, json
from typing import Dict, Any
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    print('⚠️ Set GOOGLE_API_KEY in your environment to run live calls.')


## 2) Helper: Safe API Generate Function with Retries

In [ ]:
from google import genai
import time, random

def generate(prompt: str, *, model: str='gemini-2.5-flash', temperature: float=0.7, max_tokens: int=200, retries: int=3, backoff: float=0.8) -> str:
    """Call the chat completion API with basic retries and timing.
    Returns the model's answer as plain text.
    """
    client = genai.Client(api_key=GOOGLE_API_KEY)

    if not isinstance(prompt, str):
        raise ValueError("Prompt must be a string.")
    
    for attempt in range(retries + 1):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
            )
            return response.text
        except Exception as e:
            if attempt == retries:
                raise
            sleep_time = (2 ** attempt)
            print(f"Attempt {attempt + 1} failed: {e}. Retrying in {sleep_time:.2f} seconds...")
            time.sleep(sleep_time)
    raise RuntimeError("Failed to get a response after multiple attempts.")

my_input = ""
while(my_input.strip() == "exit"):
    my_input = input("Prompt: ")
    print("Response:", end=" ")
    print(generate(my_input, temperature=0.5))





## 3) Compare Parameters

In [ ]:
prompt = 'Write three product taglines for a note-taking app.'

try:
    out1 = generate(prompt, temperature=0.2)
    out2 = generate(prompt, temperature=0.9)
    print('— Low temperature (0.2):\n', out1)
    print('\n— High temperature (0.9):\n', out2)

except Exception as e:
    raise # one would take the opportunity to do something more meaningful...

## 4) Add usage and latency metadata

Modify the function above so that it returns usage and latency.

In [ ]:
def generate(prompt: str, *, model: str='gpt-4o-mini', temperature: float=0.7, max_tokens: int=200, retries: int=3, backoff: float=0.8) -> Dict[str, Any]:
    """Call the chat completion API with basic retries and timing.
    Returns dict with text, usage (if available), and latency_ms.
    """
    ...

out = generate(prompt)
print('Usage:', json.dumps(out['usage']))
print('Latency (ms):', out['latency_ms'])

## 5) More exercises

1. Modify `generate` to accept a list of messages (system, user) and a stop sequence.
2. Add structured logging (JSON lines).
3. Try two prompts and compare responses for tone and length.